# Invoice extraction experiments — reproducible comparison

This notebook recreates the report tables from `benchmark_metrics.json`. It does not make paid API calls. The companion Kaggle notebook `reproduce_finetune_and_benchmark.ipynb` can retrain/evaluate the 0.6B adapter when supplied with the private Kaggle datasets.

Run from this artifact folder with Python 3.10+; only the Python standard library is required.


In [1]:
import json
from pathlib import Path
import csv

ROOT = Path.cwd()
if ROOT.name != "invoice-extraction-experiments":
    ROOT = ROOT / "artifacts" / "invoice-extraction-experiments"
metrics = json.loads((ROOT / "benchmark_metrics.json").read_text())


In [2]:
import importlib.util
spec = importlib.util.spec_from_file_location("saved_scoring", ROOT / "score_saved_results.py")
scoring = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scoring)
baselines = json.loads((ROOT / "inputs/baseline_models.json").read_text())
recomputed = {}
for provider, records in baselines["providers"].items():
    _, recomputed[provider] = scoring.score(list(records.values()))
for name in ("groq20b", "groq120b"):
    saved = json.loads((ROOT / "inputs" / (name + ".json")).read_text())
    _, recomputed[name] = scoring.score(list(saved["results"].values()))
    assert recomputed[name]["sample_count"] == 50
    assert recomputed[name]["whole_invoice_exact_rate"] == saved["metrics"]["whole_invoice_exact_rate"]
print({name: round(value["whole_invoice_exact_rate"] * 100, 1) for name, value in recomputed.items()})


{'modal': 10.0, 'gemini': 76.0, 'qwen17b': 44.0, 'groq20b': 50.0, 'groq120b': 74.0}


In [3]:
rows = metrics["comparison"]
columns = ["Model", "N", "Whole invoice exact (%)", "Item row exact (%)", "Mean seconds", "Latency note"]
frame = [{"Model": r["model"], "N": r["sample_count"], "Whole invoice exact (%)": round(100*r["whole_invoice_exact"],1), "Item row exact (%)": round(100*r["item_row_exact"],1), "Mean seconds": round(r["mean_seconds"],2), "Latency note": r["latency_kind"]} for r in rows]
for row in frame:
    print(f'{row["Model"]}: N={row["N"]}; whole={row["Whole invoice exact (%)"]}%; items={row["Item row exact (%)"]}%; mean={row["Mean seconds"]}s')


Qwen3-0.6B fine-tuned (Modal): N=50; whole=10.0%; items=96.1%; mean=22.33s
Qwen3-1.7B base: N=50; whole=44.0%; items=90.7%; mean=28.38s
gemini-3.5-flash-lite: N=50; whole=76.0%; items=96.1%; mean=17.45s
Groq GPT-OSS 20B; original split prompt: N=50; whole=50.0%; items=96.1%; mean=32.61s
Groq GPT-OSS 120B; complete-evidence prompt v2: N=50; whole=74.0%; items=96.1%; mean=27.69s
Groq Qwen3.8-27B; earlier prompt: N=22; whole=54.5%; items=92.0%; mean=2.17s


In [4]:
print("OCR cache SHA-256:", metrics["metadata"]["ocr_sha256"])
print("OCR:", metrics["ocr"])
print("Fine-tune:", json.dumps(metrics["fine_tune"], ensure_ascii=False, indent=2))
print("Partial v3 (excluded from final model choice):", json.dumps(metrics["groq_prompt_v3_partial"], ensure_ascii=False, indent=2))


OCR cache SHA-256: d1b9f1231cab317e4132341385b73f5644287d9a4f5101c8d30498a00713feb0
OCR: {'pipeline': 'Docling 2.129.0 + fine-tuned VietOCR vgg_seq2seq; image_max_width=512; greedy decoding.', 'model': 'lenhanm1/enterprise-printed-ocr-vietocr-tesseract', 'revision': '6deb1cb10ecbf240aead77525e80aa4c58adfb23', 'device': 'Kaggle CUDA', 'success': '50/50', 'total_seconds': 739.98, 'cache_sha256': 'd1b9f1231cab317e4132341385b73f5644287d9a4f5101c8d30498a00713feb0'}
Fine-tune: {
  "dataset_id": "ltrngnhn/invoice-synthetic-1000-sft-benchmark",
  "title": "1000 Synthetic Vietnamese Invoice SFT and Holdout",
  "split": {
    "train_invoices": 900,
    "validation_invoices": 50,
    "test_invoices": 50,
    "train_prompt_target_pairs": 1800,
    "validation_prompt_target_pairs": 100
  },
  "training_evidence": "Synthetic OCR-like header and table evidence reconstructed from labels (two tasks per train invoice); clean evidence, not OCR-error-augmented.",
  "base_model": "Qwen/Qwen3-0.6B",
  "base

In [5]:
with (ROOT / "reproduced_comparison.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows(frame)
print("Wrote", ROOT / "reproduced_comparison.csv")


Wrote /home/lenhanm1/orca/projects/my server/enterprise-knowledge-ai-agent/artifacts/invoice-extraction-experiments/reproduced_comparison.csv
